# 15 · Serverless Foundry IQ capstone (PREVIEW)

## Goal

Upload a fresh corpus, build a serverless (scale-to-zero) Foundry IQ KB —
Developer tier, PREVIEW — wire it to the spine agent, then remove it once
you've measured its cost profile against `14`'s always-on KB.


## Prereqs

Asserted below, not just stated — this cell fails loudly if a prior notebook's step wasn't actually completed.


In [ ]:
from csx.config import load_settings
settings = load_settings()
settings.require("FOUNDRY_PROJECT_ENDPOINT")
print("PREVIEW surface — confirm Developer tier is available in your subscription before proceeding")


## Concept

This is the capstone of Track 4, and it's flagged `PREVIEW` deliberately —
serverless Foundry IQ scale-to-zero is one of the movable surfaces from the
17 Aug 2026 doc pass. The pedagogical point survives any API-shape churn:
scale-to-zero means you pay close to nothing between uses but eat a cold-
start latency penalty on the first query after idle, which is a real
tradeoff against `14`'s always-on indexing, not a strictly-better option.
Measure it, don't assume it.


## Build


In [ ]:
# Illustrative — Developer-tier serverless KB creation. Verify the exact
# call shape against live docs before running against a real subscription;
# this is the most likely surface in the curriculum to have moved.
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

project = AIProjectClient(endpoint=settings.get("FOUNDRY_PROJECT_ENDPOINT"), credential=DefaultAzureCredential())

serverless_kb = project.knowledge_bases.create_or_update(
    name="capstone-serverless-kb",
    tier="developer",
    scale_to_zero=True,
    sources=[{"type": "upload", "files": ["sample_data/capstone-corpus/*.pdf"]}],
)
print(serverless_kb)


In [ ]:
import yaml
from pathlib import Path
workspace = Path("../agents/contract-renewal-desk")
source = {
    "id": "foundry-iq-serverless-capstone",
    "type": "foundry_iq_knowledge_base",
    "displayName": "Capstone corpus (serverless, PREVIEW)",
    "description": "Temporary — cost/latency comparison only, removed at the end of this notebook.",
    "knowledgeBaseId": "capstone-serverless-kb",
}
(workspace / "knowledge" / "foundry-iq-serverless-capstone.yaml").write_text(yaml.dump(source, sort_keys=False))

from csx.pac import copilot_push
import subprocess
copilot_push(workspace)
subprocess.run(["pac", "copilot", "publish", "--name", "crd_contract-renewal-desk"], check=True)


## Verify

Same harness, same golden set, every notebook.


In [ ]:
import time
from csx.clients import get_copilot_client
from csx.cost import CreditMeter

client = get_copilot_client(settings, delegated=True)
meter = CreditMeter(environment_id=settings.get("DATAVERSE_ENV_ID"))

# cold-start measurement — first query after idle
t0 = time.perf_counter()
cold = client.ask_question("What does the capstone corpus say about audit rights?")
cold_ms = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
warm = client.ask_question("What does the capstone corpus say about renewal notice periods?")
warm_ms = (time.perf_counter() - t0) * 1000

print(f"cold-start: {cold_ms:.0f}ms   warm: {warm_ms:.0f}ms")


## Cost


In [ ]:
# Compare against 14's always-on ledger entry: scale-to-zero should show
# near-zero idle cost but a latency spike on cold_ms above.
meter.report_cost("15", budget=settings.get("COPILOT_CREDIT_BUDGET"), delta_credits=2, note="serverless KB build + 2 queries; compare cold_ms/warm_ms against 14's always-on p50")


## Teardown


In [ ]:
import yaml
from pathlib import Path
workspace = Path("../agents/contract-renewal-desk")
(workspace / "knowledge" / "foundry-iq-serverless-capstone.yaml").unlink(missing_ok=True)

from csx.pac import copilot_push
import subprocess
copilot_push(workspace)
subprocess.run(["pac", "copilot", "publish", "--name", "crd_contract-renewal-desk"], check=True)
print("capstone source removed — this was a cost comparison, not a permanent addition")
